<a href="https://colab.research.google.com/github/matti410/Trading-System-Creator_V4/blob/main/Trading_Grid_Search_v4_backtesting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ricerca di Trading System

Serve a **scoprire se un segnale di ingresso ha un vantaggio reale**, separando entry, uscita e "quando sto fermo" — cercarli insieme confonde quale dei tre sta davvero funzionando.

Si legge dall'alto in basso, senza celle facoltative: ogni sezione usa il risultato di quella prima.

**Ordine:** la Sessione 1 misura il segnale puro (nessun capitale, nessuna posizione). La Sessione 2 sceglie come uscirne e diventa un sistema vero, con trade e costi.

In [ ]:
!pip install -q TA-Lib==0.8.1 backtesting==0.6.6   # versioni bloccate: cambiarle solo insieme al notebook di collaudo

### Sezione 0 · Preparazione

Clona la repo e prepara l'ambiente. Va eseguita una volta sola a inizio sessione Colab.

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import sys
from pathlib import Path
from google.colab import userdata
import shutil
# Salva il token una volta in Colab: icona chiave a sinistra → "Secrets" → aggiungi GITHUB_TOKEN
# Assicurati che il nome del secret sia 'GITHUB_TOKEN' (o quello che preferisci) e non il token stesso.
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

REPO_DIR = Path('/content/Trading-System-Creator_V4')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)  # ripulisce il tentativo fallito precedente

!git clone https://{GITHUB_TOKEN}@github.com/matti410/Trading-System-Creator_V4.git

%cd Trading-System-Creator_V4

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import datetime as dt
from dateutil.relativedelta import relativedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import talib as ta

import entry_long, entry_short
import exit_long, exit_short
import engine as eng
from engine.vwap_ops import vwap_anchored_daily
import engine.event_study
import engine.splitting
from engine.exit_search_bt import run_exit_search_bt
import filter_conditions, vwap_regime_filter_conditions
from engine.filter_search_bt import run_filter_search_bt
from engine.due_meta import due_meta
from engine.equity_plot import plot_equity
from engine.costi import (parametri_backtest, verifica_da_storico_deals,
                          deriva_costo_nel_tempo, spread_corrente)
from engine.giudizio import scheda_strategia, soglia_rumore
from engine.broker_tz_diagnostic import to_utc_index
from engine.quarantena import verifica_indice_utc, quarantena
from engine.sessioni import in_sessione, barre_da_apertura, elenco_sessioni
from entry_metro import registra_trigger_metro, NOMI_METRO
from engine.indicatori import aggiungi_indicatori
from engine.confidenza import (congela_filtro, esiti_trade, confidenza_rolling,
                               registra_filtro_confidenza,
                               report_calibrazione, plot_oos_confidenza)

## Sezione 1 · I dati

Scarica le candele da MetaTrader 5.

**Parametri che puoi cambiare:** strumento, timeframe, periodo storico.

In [ ]:
symbol    = "BTCUSD"
timeframe = 'M15'
FREQ      = "15min"      # come pandas scrive il timeframe: "15min", "h", "D"

df = pd.read_csv(f'{symbol}_M15.csv', index_col='Date')
df = to_utc_index(df, "A_US_DST (NY+7h)", on_dst_gap="drop")

print(f"{symbol} {timeframe} — {len(df):,} barre  ·  da {df.index[0]}  a  {df.index[-1]}")
df.tail(3)

### Strato 1-2 — quarantena e sessioni

`verifica_indice_utc` si ferma se l'indice non è UTC vero (difesa contro
l'etichetta fantasma). `quarantena` marca le barre non affidabili: la
finestra del rollover 16:45-18:00 NY, dove i prezzi bid sono deformati
dall'allargamento dello spread, e la prima barra dopo ogni interruzione.

Attese su EURUSD M15: ~5,2% di barre in quarantena. Se la percentuale
è molto diversa, fermati e controlla l'indice prima di proseguire.

In [ ]:
verifica_indice_utc(df)
q = quarantena(df)

display(elenco_sessioni())

In [ ]:
costi = parametri_backtest(symbol, bars=df, valuta_conto="USD", percentile=75.0, usa_mt5 = False, spread_pips = 0.1,)
costi   # -> {'spread': ..., 'commission': ...}

### Entry-metro — il righello temporale

Cinque ingressi che entrano all'apertura di una sessione e basta, senza
nessuna condizione di prezzo. **Non sono strategie: sono un metro.**

| Metro | Apertura |
|---|---|
| `M_TOKYO` | 09:00 Asia/Tokyo |
| `M_LONDRA` | 08:00 Europe/London |
| `M_NEW_YORK` | 08:00 America/New_York |
| `M_NYSE` | 09:30 America/New_York |
| `M_FIX_LONDRA` | 16:00 Europe/London |

Servono a rispondere a una domanda sola: **il mio pattern di prezzo fa
meglio del semplice entrare a quell'ora?** Un pattern che non batte il
proprio metro non sta aggiungendo informazione di prezzo: sta scegliendo
l'orologio.

#### Come si legge il segno

La curva è `(Close[i+k] / Open[i+1] - 1) × direction`, in percentuale.
Con `direction = +1` (long):

- **valore positivo** → da quell'ora in avanti il prezzo mediamente **sale**
- **valore negativo** → mediamente **scende** (quindi sarebbe uno short)

Due avvertenze che cambiano la lettura:

1. **L'ingresso è alla barra dopo.** Il trigger scatta sulla barra di
   apertura, ma si entra all'apertura della barra successiva — la regola
   anti-lookahead del framework, uguale per tutte le entry. `M_LONDRA`
   misura quindi il movimento dalle **08:15** di Londra, non dalle 08:00.
2. **Conta lo scostamento dalla linea nera del mercato, non l'altezza.**
   Su un asset con deriva la curva grezza sale da sola: guarda
   `vs_mercato_pct`.

#### Cosa NON sono

Se una metro esce con `volte_incertezza` alto, **non è un edge**: è deriva
intraday dell'asset. Le ampiezze sono di 1-2 pips, dentro il rumore dei
costi.

E soprattutto: **non vanno contate fra le prove.** Quando passi `k` a
`soglia_rumore`, passa il numero di entry *vere*, non il numero di righe
della tabella. `NOMI_METRO` serve a escluderle anche dalla ricerca delle
uscite.

In [ ]:
registra_trigger_metro()

ev_metro = engine.event_study.run_event_study(df, entry_names=list(NOMI_METRO), horizon=48)
display(ev_metro.sintesi)
ev_metro.plot()

## Sezione 2 · Gli indicatori

Le condizioni di ingresso (`entry_long.py`, `entry_short.py`) non calcolano niente da sole: leggono colonne già pronte in questa tabella. Qui le costruiamo.

| Colonna | Cos'è | Chi la usa |
|---|---|---|
| `rsi` | oscillatore 0-100: sotto 30 il prezzo è "sceso troppo", sopra 70 "salito troppo" | l'entry mean reversion |
| `ema20`, `ema50` | medie mobili veloce e lenta | l'entry di controllo |
| `zlema50` | media mobile a ritardo ridotto | l'entry di trend |
| `atr`, `realized_vol` | quanto si muove il prezzo di solito | dimensionamento futuro |
| `vwap` | prezzo medio della giornata pesato per volume | i filtri di contesto |

Sulla **zero-lag EMA**: una media normale è sempre in ritardo; la ZLEMA lo compensa proiettando in avanti il prezzo, ma esagera i movimenti e genera più falsi attraversamenti — su M15 va bene (più campioni), ma è dove i costi di transazione pesano di più.

In [ ]:
df = aggiungi_indicatori(df)
print(f"{len(df):,} barre pronte, {df.shape[1]} colonne")
print(f"adx  — min {df['adx'].min():.1f}  mediana {df['adx'].median():.1f}  max {df['adx'].max():.1f}")
print(f"atr  — mediana {df['atr'].median():.6f}  ({df['atr'].median()/df['Close'].median()*100:.3f}% del prezzo)")
df.tail(3)

## Sezione 3 · Dividere i dati in due

Questa è la regola più importante del notebook, e l'unica che non si può violare.

- **In-Sample (80%)** — la parte su cui si cerca, si prova, si sbaglia quante volte si vuole
- **Out-of-Sample (20%)** — la parte che **non si guarda fino alla fine**

Perché conta: su mille combinazioni provate, qualcuna sembrerà eccellente per puro caso — è garantito, anche su dati casuali. L'unico modo per accorgersene è tenere da parte dati che nessuna combinazione ha mai visto.

Se la guardi, cambi qualcosa e riguardi, l'Out-of-Sample non esiste più: è diventata parte della ricerca. **Si apre una volta sola, a ricerca in-sample conclusa** (passo 6 di `ROADMAP_RICERCA.md`).

In [ ]:
IS_RATIO = 0.8
df_is, df_oos = eng.splitting.split_is_oos(df, is_ratio=IS_RATIO)
eng.splitting.describe_split(df_is, df_oos)

## Sezione 4 · Features Engineering

Registra tutti i trigger di ingresso ed uscita nel registro condiviso — va fatto prima di qualunque uso di `event_study` o `exit_search_bt`, che leggono le entry da lì.

In [ ]:
entry_long.registra_trigger_long()
entry_short.registra_trigger_short()

exit_long.registra_exit_long()
exit_short.registra_exit_short()

filter_conditions.registra_filtri()
vwap_regime_filter_conditions.registra_filtri_vwap()

## SESSIONE 1 · L'andamento del prezzo dopo il trigger

Per ogni condizione: tutte le barre in cui è vera, e dove va il prezzo dopo, fino all'orizzonte `H`. Si entra all'apertura della barra successiva al segnale (mai sullo stesso Close che l'ha generato). La curva è sempre "a favore del trade": per una condizione short, sale quando il prezzo scende.

`H` è una scelta, non una griglia: si calcola una volta sola e tutti gli orizzonti più corti si leggono sulla stessa curva.

**Nessuna occorrenza viene scartata**, nemmeno quelle scattate mentre si era già dentro un trade. Ma due trigger vicini vivono le stesse candele, quindi **non contano come due prove indipendenti**: l'incertezza ne tiene conto (dal 24/9).

## 1.1 · Le curve a confronto

Ogni linea è una condizione, asse orizzontale le barre dal trigger, verticale la variazione media in percentuale. **La linea nera tratteggiata è il mercato** (nessuna condizione): il metro di paragone — sopra di lei la curva aggiunge qualcosa, appiccicata a lei non dice niente.

Guarda la **forma**: sale e prosegue = il movimento continua; sale e si appiattisce = il vantaggio ha una scadenza (è lì l'orizzonte del trade); sale e ridiscende = quello che guadagnavi lo restituisci.

In [ ]:
H = 25
MIN_TRADES = 200

In [ ]:
# solo le entry vere: le metro sono un righello, non candidate
entry_candidate = [n for n in eng.registry.list_entries() if n not in NOMI_METRO]
ev = eng.event_study.run_event_study(df_is, entry_names=entry_candidate, horizon=H, min_trades=MIN_TRADES)
ev.plot()
#ev.plot_singola("E7_BIG_TAIL_BARS", pips=True)

## 1.2 · La sintesi
| colonna | cosa dice |
|---|---|
| `trades` | quante volte la condizione è scattata |
| `barra_picco` / `barra_picco_netto` | dove la curva è più estrema, e dove si stacca di più dal mercato (usato da `diagnosi()`) |
| `picco_pct` / `picco_pips` | il valore a quel picco — **può essere negativo**: non è un errore, è un trade che perde |
| `incertezza_pct` | quanto balla la media a quel punto. Tiene conto dei trigger vicini: per le condizioni che scattano spesso è più larga |
| `volte_incertezza` | picco diviso incertezza |
| `oltre_rumore` | **la colonna da guardare per prima**: `True` se `volte_incertezza` supera la soglia stampata sotto il grafico |
| `a_fine_pct` | dove sta il prezzo a fine orizzonte: più basso del picco = il movimento è rientrato |
| `vs_mercato_pct` | quanto il trigger si stacca dal movimento che il mercato fa comunque — vicino a zero = è il respiro dell'asset, non il segnale |
| `picco_in_coda` | `True` = il picco netto cade nell'ultimo quarto della finestra, segno di deriva casuale |

**La soglia di rumore.** Provando 52 condizioni, la prima della classifica sembra buona anche su dati casuali, e in più per ognuna si sceglie la barra migliore dell'orizzonte. La riga sotto il grafico (`soglia di rumore per … entry × picco su … barre`) dice oltre quale `|volte_incertezza|` il risultato non è più spiegabile dal caso. Conta solo le prove di *questa* chiamata: è un pavimento, non un tetto.

La tabella è ordinata per ampiezza, non per solidità: guarda sempre `oltre_rumore`, `trades` e `picco_in_coda` prima di scegliere.

In [ ]:
ev.sintesi.round(4).sort_values('picco_pips', ascending=False).head(6)

### LONG side

In [ ]:
ev.sintesi[ev.sintesi['direction'] == 1][['candidato', 'picco_pips',
                                          'picco_pct', 'volte_incertezza',
                                          'oltre_rumore']].sort_values('picco_pips', ascending=False).head(6)

In [ ]:
LONG_condition_scelta = "E22_ASIAN_RANGE_BREAKOUT"

### SHORT side

In [ ]:
ev.sintesi[ev.sintesi['direction'] == -1][['candidato', 'picco_pips',
                                           'picco_pct', 'volte_incertezza',
                                           'oltre_rumore']].sort_values('picco_pips', ascending=False).head(6)

In [ ]:
SHORT_condition_scelta = None

In [ ]:
ev.diagnosi()

# SESSIONE 2 · Le condizioni di uscita

Fin qui abbiamo **misurato** un segnale: nessun capitale, nessuna posizione, nessun costo. Da adesso costruiamo un **sistema**: si sceglie un ingresso e si cerca il modo di uscirne.

## 2.1 · Come vengono calcolati stop e target

Nessuna soglia scritta a mano: uno stop dello 0.15% è enorme su EURUSD e minuscolo su BTCUSD. Per ogni barra si misura quanto sarebbe andato in rosso e in verde un trade lì aperto e tenuto `n` barre; lo stop del trade che apri ora è il **percentile richiesto di quelle escursioni sulle ultime 500 occorrenze concluse** — si allarga da solo su un asset volatile, si stringe quando il mercato si calma. Il valore si fissa all'apertura e non cambia più.

> **Perché "concluse" e non "partite".** L'escursione di un trade aperto alla barra `i` si conosce solo alla `i+n`. Usare le ultime 500 partite userebbe prezzi futuri per uno stop deciso oggi — la finestra è quindi spostata indietro di `n` barre.


## 2.2 · Backtest con i trigger selezionati

Una riga per combinazione. `combinazione` si legge `L=<entry long> · S=<entry short>`
(`—` dove quel lato non ha ingresso). Sotto `min_trades` le righe spariscono: se una
combinazione attesa non c'è, controlla prima lì.

| colonna | cosa guardi |
|---|---|
| `avg_trade_netto` | **il numero che conta**: pips medi per trade al netto delle commissioni. Sopra zero o non se ne parla |
| `t_stat` | quanto `avg_trade_netto` si distingue da zero, rispetto a quanto ballano i trade |
| `oltre_rumore` | `True` se `t_stat` supera la soglia stampata sopra la tabella |
| `avg_trade` | lo stesso, lordo — serve solo a vedere quanto si mangiano i costi |
| `trades` | sotto il centinaio, qualunque numero è rumore |
| `sharpe`, `max_dd_pct`, `win_rate_pct`, `profit_factor` | contorno |

**La soglia di rumore.** La riga sopra la tabella (`soglia di rumore per … righe`) dice oltre quale `|t_stat|` la riga migliore non è più spiegabile dal caso, contando le righe di *questa* chiamata. È un pavimento, non un tetto.

**Cosa fai dopo:** scegli la combinazione con `avg_trade_netto` più alto e abbastanza
trade, e congelala. In una chiamata l'uscita è identica per tutte le righe: la tabella
dice *quale entry rende di più con QUESTA uscita*, non quale uscita è la migliore.
Per confrontare uscite diverse rilanci con altri valori.

La tabella è ordinata per `sharpe`, non per `avg_trade_netto`: leggila per colonna, non per posizione.

**TEST 1**

Condizioni di uscita: STOP LOSS, TAKE PROFIT

In [ ]:
es = run_exit_search_bt(df_is, entry_cols_long=[LONG_condition_scelta, None],
                        entry_cols_short=[SHORT_condition_scelta, None],
                        n_barre=H, perc_sl=90.0, perc_tp=0.0,
                        **costi,)
es.top(10)
#es.trades("L=E7_BIG_TAIL_BARS · S=E9_SHORT_CLOSING_PATTERN_ONLY_II")

**TEST 2**

Condizioni di uscita: STOP LOSS, TAKE PROFIT, Regole meccaniche

In [ ]:
exit_da_provare = eng.registry.list_exit_pairs()
exit_list=[]
for i in exit_da_provare:
  exit_list.append(i[0])
exit_list

In [ ]:
es = run_exit_search_bt(df_is,
                        entry_cols_long=[LONG_condition_scelta],
                        entry_cols_short=[SHORT_condition_scelta],
                        n_barre=H,
                        exit_rule_pairs=[*exit_list],
                        perc_sl=90.0, perc_tp=0.0, **costi,
                        min_trades=30,)
es.top(10)

In [ ]:
es.top().iloc[2][0]
#es.top().iloc[2]["combinazione"] # Pandas 3.0

# SESSIONE 3

## 3.1 · La classifica

Baseline (nessun filtro) + una riga per filtro, ordinata per `t_guadagno`.

| colonna | cosa guardi |
|---|---|
| `t_guadagno` | **la colonna che decide**: i trade che il filtro TIENE battono quelli che SCARTA, e di quanto rispetto al rumore |
| `oltre_rumore` | `True` solo se `t_guadagno` supera la soglia stampata sopra la tabella |
| `guadagno_pips` | di quanto li battono, in pips — la dimensione economica |
| `esito` | `SCARTATA` · `CAMPIONE CORTO` · `DA VALUTARE` |
| `motivo` | quale criterio è caduto |
| `avg_trade_netto`, `t_stat`, `p_ev_neg` | il sistema filtrato preso da solo |

**La soglia di rumore.** Provando 17 filtri, il migliore dei 17 sembra buono anche su
dati casuali. La riga stampata sopra la tabella dice oltre quale `|t_guadagno|` il
risultato non è più spiegabile dal caso. Conta solo le prove di *questa* chiamata:
è un pavimento, non un tetto.

**Cosa fai dopo:** vanno a `due_meta` solo le righe che NON sono `SCARTATA`.
Se non ce n'è nessuna, la risposta è che questa famiglia di filtri non aggiunge
niente: si torna indietro, non si abbassa la soglia.

`guadagno_sharpe` resta in `fs.risultati` come riferimento. Non decide più —
è uno Sharpe di equity giornaliera, quindi dipende da quanti trade fai e da quanto
stai a mercato, non solo dalla qualità del vantaggio.

In [ ]:
filtri_da_provare = (eng.registry.list_filters(0) +
                     eng.registry.list_filters(1) +
                     eng.registry.list_filters(-1))

In [ ]:
fs = run_filter_search_bt(
    df_is,
    entry_long=LONG_condition_scelta,
    entry_short=SHORT_condition_scelta,
    exit_rule_pair=None,
    n_barre=H,
    filtri=filtri_da_provare,
    perc_sl=90.0, perc_tp=0.0, **costi, min_trades_giudizio=100,
    p_max=5.0, alpha=0.05
)
fs.top(10)

In [ ]:
#fs.scheda()                      # la baseline
fs.scheda("F8_UPTREND_CONTEXT")  # una riga filtrata

In [ ]:
lista_etichette_filtri_dm = (fs.risultati
    .loc[fs.risultati['esito'].isin(['DA VALUTARE', 'CAMPIONE CORTO']), 'filtro']
    .to_list())
print(f"{len(lista_etichette_filtri_dm)} candidati: {lista_etichette_filtri_dm}")

## 3.2 · `due_meta` — il vantaggio regge su entrambe le metà?

Si spacca l'In-Sample a metà e si guarda se il filtro aggiunge in tutte e due
(Difesa C, Passo 5). Un filtro che funziona solo in una metà è rumore.

Entry, uscita, orizzonte, `t_guadagno` e soglia si leggono **da `fs`**: non è
possibile verificare per errore un setup diverso. Il resto (`perc_sl`, `perc_tp`,
costi) va ripassato identico alla cella che ha costruito `fs`.

**Due condizioni, per due ragioni diverse:**

1. **Stabilità** — `guadagno_pips` positivo in entrambe le metà.
2. **Significatività** — `t_guadagno` sopra soglia sull'**intero** In-Sample.

La significatività si giudica sull'intero perché con ~250 trade per metà il rumore
produce da solo oscillazioni enormi. Le metà servono a scoprire un **cambio di
segno**, non a misurare con precisione. Per questo un calo fra le due metà non è
di per sé una condanna: conta il segno.

| verdetto | quando |
|---|---|
| `regge` | positivo in entrambe **e** t sopra soglia |
| `giudizio sospeso` | come sopra, ma poche osservazioni in una metà |
| `non regge` | `motivo` dice quale condizione è caduta |
| `baseline` | riga di riferimento (`None` fra le etichette), non un verdetto |

**Cosa fai dopo:** all'Out-of-Sample vanno solo i `regge`. Se non ce n'è nessuno,
non si apre l'OOS.

In [ ]:
assert fs.n_barre == H
etichette = [None, *lista_etichette_filtri_dm]  # 2-3 nomi presi da fs.top(15), es. "TREND_CONTEXT", "F1_ADX_ABOVE"

In [ ]:
dm = due_meta(df_is, fs, etichette, perc_sl=90.0, perc_tp=0.0, **costi)
dm.tabella

In [ ]:
lista_etichette_filtri_oos = dm.tabella[dm.tabella['verdetto'] == 'regge']['filtro'].to_list()
lista_etichette_filtri_oos

In [ ]:
filtro_scelto = 'F8_UPTREND_CONTEXT'     # l'etichetta della riga scelta in fs.top()

# il filtro entra nell'entry: da qui in poi il setup è «entry + filtro»
entry_long_cong, entry_short_cong = congela_filtro(fs, filtro_scelto)
print(f"setup congelato: long = {entry_long_cong} · short = {entry_short_cong}")



------------------------------------------------------------------------------------------------------------------------


------





## 3.3 · La confidenza dei segnali

Per ogni segnale, la confidenza è **come sono andati i trade passati caduti nello
stesso contesto di mercato**. Il contesto è una griglia 2×2:

| | trend contro il trade | trend a favore del trade |
|---|---|---|
| **volatilità bassa** | VOL_BASSA·TREND_CONTRO | VOL_BASSA·TREND_FAVORE |
| **volatilità alta** | VOL_ALTA·TREND_CONTRO | VOL_ALTA·TREND_FAVORE |

- volatilità: std dei rendimenti su 20 barre, sopra o sotto la mediana delle ultime 500
- trend: `aura_gap` (regressione intraday − VWAP), girato nel verso del trade, sopra o sotto la mediana delle ultime 500

**Niente futuro.** Un trade entra nella storia solo dalla barra in cui è USCITO. La storia viene dal setup congelato — entry + filtro_scelto — fatto girare su tutta la serie: la
confidenza di una barra usa solo trade chiusi prima di lei, quindi anche sull'OOS
è calcolata come si farebbe dal vivo. Qui sotto non si mostra niente dell'OOS.

**Come si usa.** Una soglia sulla confidenza è un filtro: «opero solo se i pips
netti medi dei trade simili sono sopra X». Ogni soglia è una riga della solita
tabella dei filtri, con le solite difese: tenuti vs scartati, soglia di rumore,
`due_meta`. Dove la confidenza non è ancora nota (meno di 30 trade nella cella) il
segnale non si trada.

In [ ]:
# 1. la storia: il setup congelato (entry + filtro scelto) su tutta la serie
base_completa = run_filter_search_bt(
    df, entry_long=entry_long_cong, entry_short=entry_short_cong,
    exit_rule_pair=fs.exit_rule_pair, n_barre=fs.n_barre, filtri=[],
    perc_sl=90.0, perc_tp=0.0, **costi, verbose=False)
esiti = esiti_trade(base_completa.trades(), df, fs.pip_size, costi["commission"])

# 2. la confidenza barra per barra, per ciascun lato attivo
conf_long  = confidenza_rolling(df, esiti, direzione=+1) if entry_long_cong  else None
conf_short = confidenza_rolling(df, esiti, direzione=-1) if entry_short_cong else None

print(f"{len(esiti)} trade nella storia · lato long {int((esiti['lato'] == 1).sum())}"
      f" · lato short {int((esiti['lato'] == -1).sum())}")

In [ ]:
# confidenza prevista (prima di entrare) contro esito reale, per cella
report_calibrazione(esiti, df, split=df_oos.index[0],
                    conf_long=conf_long, conf_short=conf_short).round(3)

In [ ]:
SOGLIE_CONF = [0.0, 0.5, 1.0]     # pips netti medi: si opera se la confidenza è SOPRA

filtri_conf = registra_filtro_confidenza(SOGLIE_CONF,
                                         conf_long=conf_long, conf_short=conf_short)

fs_conf = run_filter_search_bt(
    df_is,
    entry_long=entry_long_cong, entry_short=entry_short_cong,
    exit_rule_pair=fs.exit_rule_pair, n_barre=fs.n_barre,
    filtri=filtri_conf,
    perc_sl=90.0, perc_tp=0.0, **costi, min_trades_giudizio=100,
    p_max=5.0, alpha=0.05)
fs_conf.top(10)

In [ ]:
cand_conf = (fs_conf.risultati
    .loc[fs_conf.risultati['esito'].isin(['DA VALUTARE', 'CAMPIONE CORTO']), 'filtro']
    .to_list())
print(f"{len(cand_conf)} candidati: {cand_conf}")

dm_conf = due_meta(df_is, fs_conf, [None, *cand_conf], perc_sl=90.0, perc_tp=0.0, **costi)
dm_conf.tabella

# SESSIONE 4 · Out-of-Sample (Passo 6)

Qui si apre il tratto di dati mai guardato finora — sigillato fin dallo split di Sessione 3 (`df_is`, `df_oos`). Non è un altro giro di ricerca: è la verifica finale, **una volta sola**, sui candidati che in `due_meta` hanno verdetto `regge`.

**Perché una volta sola.** Ogni volta che si guarda l'Out-of-Sample e si cambia qualcosa — un parametro, la lista dei filtri, la soglia — quel tratto di dati smette di essere una verifica indipendente e diventa parte della ricerca stessa, esattamente come l'In-Sample. Se il risultato di questa cella non convince, la risposta non è "aggiustare e rilanciare": è tornare al Passo 5 con un'idea diversa, sapendo che questo OOS specifico non è più vergine.

**Setup ed entry non si ripassano: si leggono da `fs`.** Come `due_meta`, anche questa cella legge `entry_long`, `entry_short`, `exit_rule_pair`, `n_barre`, `min_trades` direttamente dagli attributi di `fs` — non è possibile testare per errore un setup diverso da quello che `fs.top()` e `dm.tabella` hanno già mostrato.

**Il resto va ripassato identico** alla cella che ha costruito `fs` e a quella di `due_meta`: `perc_sl`, `perc_tp`, `finestra`, `lag`, `spread`, `commission`, `cash`, `margin` — non sono salvati su `fs`, quindi vanno riscritti a mano uguali a sopra.

**I candidati** sono la lista `filtri_oos`, presa a mano da quali righe di `dm.tabella` hanno verdetto `regge` — va aggiornata qui se cambiano le promozioni.

**Come si legge `oos.risultati`:** stesse colonne di `fs.top()` — baseline + una riga per candidato.

| colonna | cosa guardi |
|---|---|
| `avg_trade_netto` | pips per trade sull'OOS, al netto dei costi |
| `t_guadagno` | deve avere **lo stesso segno** dell'In-Sample e un ordine di grandezza simile. Un ribaltamento di segno è la firma della selezione a posteriori |
| `esito` | come in Sessione 3 |
| `trades` | per forza più bassi: l'OOS è il 20% dei dati |

**Prima cosa da guardare, sempre: la riga baseline.** Se il setup congelato non
regge da solo su dati mai visti, il problema è a monte del filtro.

**Cosa fare dopo:** niente. Si legge il risultato, si registra il giudizio (nei materiali del progetto, non ricalcolando), e si passa oltre — in avanti verso il prossimo passo della roadmap, o indietro al Passo 5 con un'idea diversa se l'OOS non convince. Non si ritocca questa cella.

In [ ]:
# le etichette di dm.tabella vanno tradotte nei nomi registrati:
# per una coppia l'etichetta non è un filtro, sono i due membri che la compongono
_righe = fs.risultati[fs.risultati['filtro'].isin(lista_etichette_filtri_oos)]
filtri_oos = sorted({n for n in _righe[['filtro_long', 'filtro_short']]
                            .to_numpy().ravel() if n != '—'})
print(f"filtri_oos: {filtri_oos}")

oos = run_filter_search_bt(
    df_oos,
    entry_long=fs.entry_long,
    entry_short=fs.entry_short,
    exit_rule_pair=fs.exit_rule_pair,
    n_barre=fs.n_barre,
    filtri=filtri_oos,
    perc_sl=90.0, perc_tp=0.0, finestra=500, lag=1,
    **costi, cash=10_000.0, margin=1.0,
    min_trades=fs.min_trades,
)
oos.risultati

## 4.2 · Confidenza: con e senza (OOS)

Due backtest separati sull'OOS: il setup senza filtro e lo stesso setup con la soglia
di confidenza scelta in-sample. Non è la stessa lista di trade tagliata: con il
filtro entrano anche trade che senza erano bloccati da una posizione già aperta.

`CONF_SCELTA` si decide **prima** di eseguire la cella, guardando solo `fs_conf` e
`dm_conf`. Si esegue una volta sola.

In [ ]:
CONF_SCELTA = "CONF_PIPS_MEDI_0.5_L"    # l'etichetta scelta in-sample, da fs_conf

_r = fs_conf.risultati.set_index('filtro').loc[CONF_SCELTA]
filtri_conf_oos = [n for n in (_r['filtro_long'], _r['filtro_short']) if n != '—']

oos_conf = run_filter_search_bt(
    df_oos,
    entry_long=entry_long_cong, entry_short=entry_short_cong,
    exit_rule_pair=fs.exit_rule_pair, n_barre=fs.n_barre,
    filtri=filtri_conf_oos,
    perc_sl=90.0, perc_tp=0.0, finestra=500, lag=1,
    **costi, cash=10_000.0, margin=1.0, min_trades=fs.min_trades)
display(oos_conf.risultati)

ax, tabella_conf = plot_oos_confidenza(oos_conf.trades(), oos_conf.trades(CONF_SCELTA),
                                       oos_conf.pip_size, costi["commission"],
                                       etichetta_con=CONF_SCELTA)
display(tabella_conf)

report_calibrazione(esiti, df, split=df_oos.index[0], conf_long=conf_long,
                    conf_short=conf_short, mostra_oos=True).round(3)

## 4.1 · Equity vs buy & hold (OOS)

**Attenzione a come si legge:** buy & hold pieno su EURUSD non è un
benchmark "neutro" — il cambio non ha una deriva strutturale di lungo
periodo come un indice o BTC, quindi il suo andamento qui riflette
soprattutto la direzione presa dal prezzo in *questo* tratto OOS, non
un'alternativa passiva razionale. Batterlo (o perderci) non è di per sé
la prova che conta — quella resta oos.risultati (t_guadagno sopra la baseline). Questo grafico è un controllo visivo in più, non il
verdetto.

In [ ]:
for nome in [*lista_etichette_filtri_oos, None]:
    ax, curva = plot_equity(oos.trades(nome), df_oos, cash=10_000.0,
                            etichetta=f"{nome} (OOS)")